# Lab 11 — Generating images with diffusion models

**Deep Learning for Computer Vision — Day 3** · ECAM, 5MAU · 2026–2027

**Goals.**
1. Implement the **forward (noising) process** of a diffusion model and see what $q(x_t \mid x_0)$ does to an image.
2. Train a tiny **DDPM from scratch** on 2D points (runs in ~1 min on a CPU) and generate new samples with the **reverse process**.
3. *(Colab, GPU)* Use **Stable Diffusion** with `diffusers`: prompts, seeds, number of steps, guidance scale, image-to-image — and discuss synthetic data for industrial datasets.

**Duration:** ~45 min. Sections 1–2 run anywhere; section 3 needs a GPU (Colab: *Runtime → Change runtime type → T4 GPU*, ~4 GB download).

> *References:* J. Sohl-Dickstein *et al.*, "Deep unsupervised learning using nonequilibrium thermodynamics", ICML 2015 (swiss roll) · J. Ho, A. Jain, P. Abbeel, "Denoising Diffusion Probabilistic Models", NeurIPS 2020 · R. Rombach *et al.*, "High-Resolution Image Synthesis with Latent Diffusion Models", CVPR 2022 · J. Ho, T. Salimans, "Classifier-free diffusion guidance", NeurIPS 2021 Workshop · Hugging Face `diffusers` documentation, https://huggingface.co/docs/diffusers · S. J. D. Prince, *Understanding Deep Learning*, MIT Press, 2023, ch. 18.

In [ ]:
import sys

if "google.colab" in sys.modules:  # Colab: the packages are installed by the next cell
    print("Colab: setup check skipped.")
else:
    # Setup check: are all the packages of requirements.txt installed in this kernel?
    import importlib
    import importlib.util

    importlib.invalidate_caches()
    REQUIRED = {"torch": "torch", "torchvision": "torchvision", "ultralytics": "ultralytics", "numpy": "numpy",
                "matplotlib": "matplotlib", "cv2": "opencv-contrib-python"}
    missing = [pkg for module, pkg in REQUIRED.items() if importlib.util.find_spec(module) is None]
    assert not missing, (f"Missing packages: {missing}. Run `%pip install -r ../requirements.txt` "
                         "in a cell, then restart the kernel (Kernel > Restart) and run the notebook again.")
    print("Setup OK: all required packages are installed.")

In [ ]:
import sys
import time
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB and not Path("data/table_example.jpg").exists():
    from google.colab import files
    print("Upload table_example.jpg (from the data folder of the course)")
    Path("data").mkdir(exist_ok=True)
    for name, content in files.upload().items():
        Path("data", name).write_bytes(content)
DATA = Path("data") if Path("data").exists() else Path("../data")
torch.manual_seed(0)
GPU = torch.cuda.is_available()
print("Colab:", IN_COLAB, "| GPU:", GPU)

## 1. The forward process: destroying an image with noise

DDPM adds a little Gaussian noise at each of $T = 1000$ steps, with a variance schedule $\beta_1 = 10^{-4} \dots \beta_T = 0.02$. Thanks to Gaussian properties, $x_t$ can be sampled **directly** from $x_0$:
$$ x_t = \sqrt{\bar\alpha_t}\,x_0 + \sqrt{1 - \bar\alpha_t}\,\varepsilon, \qquad \varepsilon \sim \mathcal{N}(0, I), \qquad \bar\alpha_t = \prod_{s=1}^{t} (1 - \beta_s). $$
At $t = T$, $\bar\alpha_T \approx 0$: $x_T$ is pure noise.

### Exercise 1.1
Compute `alpha_bar` and implement `add_noise(x0, t, eps)`. Apply it to an image (scaled to $[-1, 1]$) for several $t$.

In [ ]:
T_STEPS = 1000
betas = torch.linspace(1e-4, 0.02, T_STEPS)
# TODO
alphas = ... ; alpha_bar = ...


def add_noise(x0, t, eps):
    """x0: tensor, t: integer step (0-based) or tensor of steps with the batch dimension first."""
    # TODO
    ...


assert torch.isclose(alpha_bar[-1], torch.prod(1 - betas)) and alpha_bar[-1] < 1e-4

img = cv2.imread(str(DATA / "table_example.jpg"))[:, :, ::-1]
x0 = torch.from_numpy(img.copy()).float().permute(2, 0, 1) / 127.5 - 1
eps = torch.randn_like(x0)
steps = [0, 50, 100, 200, 400, 700, 999]
fig, ax = plt.subplots(1, len(steps) + 1, figsize=(18, 2.6))
for a, t in zip(ax, steps):
    xt = add_noise(x0, t, eps)
    a.imshow(((xt.permute(1, 2, 0) + 1) / 2).clamp(0, 1)); a.set_title(f"t = {t}"); a.axis("off")
ax[-1].plot(alpha_bar.sqrt(), label=r"$\sqrt{\bar\alpha_t}$ (signal)")
ax[-1].plot((1 - alpha_bar).sqrt(), label=r"$\sqrt{1-\bar\alpha_t}$ (noise)"); ax[-1].legend(fontsize=7)

**Question.** The forward process has no parameter to learn. What is learned then?

## 2. A tiny DDPM from scratch

To see the whole algorithm on a CPU, our "images" are **2D points** sampled from a spiral (the "swiss roll", already used by the very first diffusion model). The denoiser is a small MLP taking $(x_t, t)$ and returning the predicted noise.

**Training** (DDPM, "simple" loss): repeat
1. take a batch $x_0$, a random step $t$ per sample, a noise $\varepsilon$;
2. build $x_t$ with `add_noise`;
3. minimise $\| \varepsilon - \varepsilon_\theta(x_t, t) \|^2$.

In [ ]:
def make_spiral(n=10000, seed=0):
    rng = np.random.default_rng(seed)
    th = np.sqrt(rng.uniform(0, 1, n)) * 3 * np.pi + np.pi
    pts = np.c_[th * np.cos(th), th * np.sin(th)] / (4 * np.pi) * 2 + rng.normal(0, 0.03, (n, 2))
    return torch.from_numpy(pts.astype(np.float32))


data = make_spiral()
print(len(data), "points")
plt.figure(figsize=(3, 3)); plt.scatter(*data[::4].T, s=1); plt.axis("equal"); plt.title("training data");

In [ ]:
class Denoiser(nn.Module):
    """eps_theta(x_t, t): a small MLP with a sinusoidal embedding of the step t."""

    def __init__(self, hidden=256, n_freq=16):
        super().__init__()
        self.freqs = torch.exp(torch.linspace(0, np.log(1000), n_freq))
        self.net = nn.Sequential(nn.Linear(2 + 2 * n_freq, hidden), nn.SiLU(),
                                 nn.Linear(hidden, hidden), nn.SiLU(),
                                 nn.Linear(hidden, hidden), nn.SiLU(),
                                 nn.Linear(hidden, 2))

    def forward(self, x, t):
        a = (t.float() / T_STEPS)[:, None] * self.freqs[None]
        return self.net(torch.cat([x, torch.sin(a), torch.cos(a)], dim=1))

### Exercise 2.1 — Training step
Complete the training loop (sample `t` and `eps`, build `xt`, compute the MSE loss).

In [ ]:
torch.manual_seed(0)
model = Denoiser()
opt = torch.optim.Adam(model.parameters(), lr=2e-3)
N_ITERS = 4000
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, N_ITERS)
losses, t0 = [], time.perf_counter()
for it in range(N_ITERS):
    x0b = data[torch.randint(len(data), (512,))]
    # TODO
    ...
    opt.zero_grad(); loss.backward(); opt.step(); sched.step()
    losses.append(loss.item())
print(f"trained in {time.perf_counter() - t0:.0f}s, final loss {np.mean(losses[-200:]):.3f}")
plt.figure(figsize=(6, 2.5)); plt.plot(np.convolve(losses, np.ones(50) / 50, "valid")); plt.xlabel("iteration"); plt.ylabel("loss");

### Exercise 2.2 — Sampling (reverse process)
Start from pure noise $x_T \sim \mathcal{N}(0, I)$ and for $t = T-1, \dots, 0$:
$$ x_{t-1} = \frac{1}{\sqrt{\alpha_t}}\Big(x_t - \frac{\beta_t}{\sqrt{1 - \bar\alpha_t}}\,\varepsilon_\theta(x_t, t)\Big) + \sqrt{\beta_t}\,z, \qquad z \sim \mathcal{N}(0, I)\ \text{(no noise at the last step)}. $$

In [ ]:
@torch.no_grad()
def sample(model, n=3000, keep=(999, 600, 300, 100, 0)):
    x = torch.randn(n, 2)
    snapshots = {}
    for t in range(T_STEPS - 1, -1, -1):
        tt = torch.full((n,), t)
        # TODO
        ...
        if t in keep:
            snapshots[t] = x.clone()
    return x, snapshots


torch.manual_seed(1)
samples, snaps = sample(model)
fig, ax = plt.subplots(1, len(snaps), figsize=(18, 2.6))
for a, (t, s) in zip(ax, snaps.items()):
    a.scatter(*s.T, s=1); a.set_title(f"t = {t}"); a.set_xlim(-3, 3); a.set_ylim(-3, 3); a.set_aspect("equal")

**Bonus.** Replace the spiral by the points of the word *ECAM* (`cv2.putText` on a black canvas, then `np.nonzero`). Even with 3× more iterations the letters stay blurry: thin structures are much harder to learn. Real image models use large U-Nets, millions of images and days of GPU training.

**Questions.**
1. The generated points are **new** points, not copies of the training points. How could you check it?
2. Why does sampling need 1 000 network evaluations, and why is that a problem? (Stable Diffusion uses 20–50 steps with better samplers.)

## 3. Stable Diffusion (Colab, GPU)

**Stable Diffusion** = a diffusion model working in the **latent space** of an auto-encoder (64×64×4 latents for 512×512 images), whose denoiser is a **U-Net conditioned on text** through cross-attention on CLIP text embeddings. **Classifier-free guidance** combines the conditional and unconditional predictions:
$$ \hat\varepsilon = \varepsilon_\theta(z_t, \varnothing) + w\,\big(\varepsilon_\theta(z_t, \text{prompt}) - \varepsilon_\theta(z_t, \varnothing)\big). $$

Set `RUN_SD = True` on a GPU runtime. The first run downloads the model (~4 GB). Accept the licence of the model on Hugging Face if asked.

In [ ]:
RUN_SD = GPU  # automatically True on a Colab GPU runtime
if RUN_SD:
    if IN_COLAB:
        !pip -q install diffusers transformers accelerate
    from diffusers import StableDiffusionImg2ImgPipeline, StableDiffusionPipeline
    pipe = StableDiffusionPipeline.from_pretrained("stable-diffusion-v1-5/stable-diffusion-v1-5",
                                                   torch_dtype=torch.float16).to("cuda")
    pipe.set_progress_bar_config(disable=True)


def show_row(images, titles):
    fig, ax = plt.subplots(1, len(images), figsize=(4 * len(images), 4))
    for a, im, t in zip(np.atleast_1d(ax), images, titles):
        a.imshow(im); a.set_title(t, fontsize=9); a.axis("off")

### Exercise 3.1 — Prompt and seed
Generate an image for the prompt below with 3 different seeds (`generator=torch.Generator("cuda").manual_seed(s)`). Then generate twice with the **same** seed: what do you observe?

In [ ]:
prompt = "top view photo of a grey metal washer and a red plastic disc on a light grey workbench, industrial lighting"
if RUN_SD:
    imgs = [pipe(prompt, num_inference_steps=25, guidance_scale=7.5,
                 generator=torch.Generator("cuda").manual_seed(s)).images[0] for s in [0, 1, 2, 0]]
    show_row(imgs, ["seed 0", "seed 1", "seed 2", "seed 0 again"])

### Exercise 3.2 — Guidance scale and number of steps
Fix the seed and vary `guidance_scale` in {1, 4, 7.5, 15}, then `num_inference_steps` in {5, 15, 30, 50}. Describe the effect of each parameter.

In [ ]:
if RUN_SD:
    g = lambda: torch.Generator("cuda").manual_seed(0)
    # TODO
    ...

### Exercise 3.3 — Image-to-image: variants of our work table
`img2img` starts from a **noised version of an existing image** instead of pure noise. `strength` (0–1) sets how much noise is added, i.e. how far the result can move from the input.

In [ ]:
if RUN_SD:
    from PIL import Image
    i2i = StableDiffusionImg2ImgPipeline(**pipe.components).to("cuda")
    init = Image.open(DATA / "table_example.jpg").convert("RGB").resize((768, 432))
    p2 = "photo of coloured plastic parts and metal washers on a workbench, realistic, sharp"
    outs = [i2i(p2, image=init, strength=s, guidance_scale=7.5, generator=torch.Generator("cuda").manual_seed(0)).images[0]
            for s in [0.2, 0.4, 0.6, 0.8]]
    show_row([init] + outs, ["input"] + [f"strength {s}" for s in [0.2, 0.4, 0.6, 0.8]])

### Discussion — synthetic data for the project
1. Could you train the YOLO detector of Lab 10 on images generated by Stable Diffusion? What would you need besides the images?
2. What is the *domain gap*, and how would you check that synthetic images help (which experiment, which metric)?
3. Give two risks or limits of generative models (technical, legal or ethical).